In [19]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_date, datediff

# create spark session
spark = SparkSession.builder.appName("DataQualityChecks").getOrCreate()

# sample data
data = [
    (1, "Ravi", 500, "2026-02-15"),
    (2, None, 300, "2026-02-16"),
    (3, "Aman", -50, "2026-01-01"),
    (4, "Neha", 700, "2025-12-01")
]

columns = ["id", "customer", "amount", "order_date"]

df = spark.createDataFrame(data, columns)
df.show()


+---+--------+------+----------+
| id|customer|amount|order_date|
+---+--------+------+----------+
|  1|    Ravi|   500|2026-02-15|
|  2|    NULL|   300|2026-02-16|
|  3|    Aman|   -50|2026-01-01|
|  4|    Neha|   700|2025-12-01|
+---+--------+------+----------+



In [20]:
# find rows where customer is missing
missing_customer = df.filter(col("customer").isNull())

missing_customer.show()


+---+--------+------+----------+
| id|customer|amount|order_date|
+---+--------+------+----------+
|  2|    NULL|   300|2026-02-16|
+---+--------+------+----------+



Explanation (comment style):

.isNull() detects missing values

Helps ensure required fields (like customer name) are present

Enterprises usually reject or fix these records

In [21]:
# find invalid transactions where amount < 0
invalid_amount = df.filter(col("amount") < 0)

invalid_amount.show()


+---+--------+------+----------+
| id|customer|amount|order_date|
+---+--------+------+----------+
|  3|    Aman|   -50|2026-01-01|
+---+--------+------+----------+



In [23]:
# convert order_date to date type
from pyspark.sql.functions import to_date

df2 = df.withColumn("order_date", to_date(col("order_date")))

# check records older than 30 days
old_records = df2.filter(datediff(current_date(), col("order_date")) > 30)

old_records.show()


+---+--------+------+----------+
| id|customer|amount|order_date|
+---+--------+------+----------+
|  3|    Aman|   -50|2026-01-01|
|  4|    Neha|   700|2025-12-01|
+---+--------+------+----------+



Explanation:

Business rule: amount must be positive

Negative values indicate entry or system error

Companies block these before billing/reporting

datediff() compares today's date and order date

Records older than 30 days are considered stale

Used in inventory, fraud detection, and reporting systems

In real pipelines:

If completeness fails → record rejected

If accuracy fails → flagged for correction

If timeliness fails → alert data team

Often implemented in daily ETL jobs before loading into a data warehouse.

In [24]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("DataQualityCheck").getOrCreate()
df = spark.read.csv("./covid-data.csv", header=True, inferSchema=True)  # Adjust path as needed

# Check completeness (nulls in critical columns)
df.filter(df.new_cases.isNull()).show()
# Output: Rows where 'new_cases' is missing (should be empty for complete data)

+--------+-------------+---------+----------+-----------+---------+------------------+------------+----------+-------------------+-----------------------+---------------------+------------------------------+------------------------+----------------------+-------------------------------+-----------------+------------+------------------------+-------------+-------------------------+---------------------+---------------------------------+----------------------+----------------------------------+-----------+---------+------------------------+----------------------+------------------+-------------------------------+-------------+--------------+---------------+------------------+-----------------+-----------------------+--------------+----------------+-------------------------+------------------------------+-----------------------------+-----------------------------------+--------------------------+-------------------------------------+------------------------------+-------------------------

In [10]:
!wget -O /home/jovyan/work/covid-data.csv \
  https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv


--2026-02-23 14:36:06--  https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 98391483 (94M) [text/plain]
Saving to: ‘/home/jovyan/work/covid-data.csv’

/home/jovyan/work/c 100%[===================>]  93.83M  23.1MB/s    in 4.1s    

2026-02-23 14:36:10 (22.8 MB/s) - ‘/home/jovyan/work/covid-data.csv’ saved [98391483/98391483]



In [26]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_date, datediff

# create spark session
spark = SparkSession.builder.appName("DataQualityChecks").getOrCreate()

df_spark = spark.read.csv("./covid-data.csv", header=True, inferSchema=True)
df_spark.show(5)

+--------+---------+-----------+----------+-----------+---------+------------------+------------+----------+-------------------+-----------------------+---------------------+------------------------------+------------------------+----------------------+-------------------------------+-----------------+------------+------------------------+-------------+-------------------------+---------------------+---------------------------------+----------------------+----------------------------------+-----------+---------+------------------------+----------------------+------------------+-------------------------------+-------------+--------------+-----------+------------------+-----------------+-----------------------+--------------+----------------+-------------------------+------------------------------+-----------------------------+-----------------------------------+--------------------------+-------------------------------------+------------------------------+-------------------------------

In [12]:
from pyspark.sql import functions as F
spark = SparkSession.builder.appName("DataQualityCheck").getOrCreate()

# Load the dataset
df = spark.read.csv("./covid-data.csv", header=True, inferSchema=True)  # Adjust path as needed

# Count the number of nulls in each column to identify missing data
null_counts = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns])
null_counts.show()  # Output: Number of missing values per column

+--------+---------+--------+----+-----------+---------+------------------+------------+----------+-------------------+-----------------------+---------------------+------------------------------+------------------------+----------------------+-------------------------------+-----------------+------------+------------------------+-------------+-------------------------+---------------------+---------------------------------+----------------------+----------------------------------+-----------+---------+------------------------+----------------------+------------------+-------------------------------+-------------+--------------+-----------+------------------+-----------------+-----------------------+--------------+----------------+-------------------------+------------------------------+-----------------------------+-----------------------------------+--------------------------+-------------------------------------+------------------------------+----------------------------------------

In [27]:
import pyspark.sql.functions as F

null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in df.columns
])

# Convert the single-row wide result into (column, null_count) rows
nulls_long = null_counts.selectExpr(
    "stack({}, {}) as (column, null_count)".format(
        len(df.columns),
        ", ".join([f"'{c}', `{c}`" for c in df.columns])
    )
)

# Show only columns that actually have nulls, sorted
(nulls_long
 .filter(F.col("null_count") > 0)
 .orderBy(F.desc("null_count"))
 .show(200, truncate=False))

+------------------------------------------+----------+
|column                                    |null_count|
+------------------------------------------+----------+
|weekly_icu_admissions                     |418442    |
|weekly_icu_admissions_per_million         |418442    |
|excess_mortality_cumulative_absolute      |416024    |
|excess_mortality_cumulative               |416024    |
|excess_mortality                          |416024    |
|excess_mortality_cumulative_per_million   |416024    |
|weekly_hosp_admissions                    |404938    |
|weekly_hosp_admissions_per_million        |404938    |
|icu_patients                              |390319    |
|icu_patients_per_million                  |390319    |
|hosp_patients                             |388779    |
|hosp_patients_per_million                 |388779    |
|total_boosters                            |375835    |
|total_boosters_per_hundred                |375835    |
|new_vaccinations                          |3584